# Colab-Portable Evaluation Notebook for HTv2/BTC Checkpoints

This notebook is the **single-file Colab version**.

It embeds the required repo source files and writes them into a temporary runtime directory when the notebook starts. That means you do **not** need to upload `dataset.py`, `models/`, or `evaluation_utils.py` separately every Colab session.

Put on Google Drive:
- this notebook,
- `data/chord_data_1217/processed/`,
- `data/chord_data_1217/chordlab/`,
- `data/chord_data_1217/splits/`,
- your checkpoint under `data/chord_data_1217/checkpoints/...`.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# Optional: install dependencies into the current Colab runtime.
!pip install -q numpy pandas tqdm mir_eval jams matplotlib seaborn torch


In [ ]:

import json
import sys
from pathlib import Path

# Adjust this to your Google Drive layout.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/ML Project').resolve()
assert DRIVE_PROJECT_DIR.exists(), f'Missing project dir: {DRIVE_PROJECT_DIR}'

RUNTIME_DIR = Path('/content/ht_eval_runtime').resolve()
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
(RUNTIME_DIR / 'models').mkdir(parents=True, exist_ok=True)


In [ ]:
EMBEDDED_FILES = {
    "dataset.py": "import os\nimport json\nfrom dataclasses import dataclass\nfrom typing import Dict, List\nimport random\nimport re\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset, DataLoader, DistributedSampler, Sampler\n\n\nTARGET_CLASSES = [\n    \"Others\",\n    \"min/b7\",\n    \"min/2\",\n    \"maj/b7\",\n    \"maj/2\",\n    \"sus4(b7)\",\n    \"sus2\",\n    \"sus4\",\n    \"13\",\n    \"11\",\n    \"min9\",\n    \"9\",\n    \"maj9\",\n    \"dim7\",\n    \"hdim7\",\n    \"min7\",\n    \"7\",\n    \"maj7\",\n    \"min/5\",\n    \"min/b3\",\n    \"maj/5\",\n    \"maj/3\",\n    \"dim\",\n    \"aug\",\n    \"min\",\n    \"maj\",\n    \"N\",\n]\n\n@dataclass\nclass ProcessedChordConfig:\n    root_dir: str\n    n_steps: int = 128\n    stride: int = 64\n    batch_size: int = 8\n    num_workers: int = 0\n    window_mode: str = \"sliding\"\n    label_mode: str = \"quality27\"\n    augment_train: bool = False\n    noise_std: float = 0.01\n    gain_min: float = 0.9\n    gain_max: float = 1.1\n    time_mask_width: int = 8\n    freq_mask_width: int = 12\n    pitch_shift_bins: int = 0\n    pitch_shift_semitones: int = 0\n    use_signal_decay: bool = False\n    signal_decay_min: float = 0.4\n    signal_decay_max: float = 0.9\n    distributed: bool = False\n    rank: int = 0\n    world_size: int = 1\n\n\nclass ChordVocab:\n    def __init__(self, labels: List[str], label_mode: str = \"full_chord\"):\n        uniq = sorted(set(labels))\n        if \"N\" not in uniq:\n            uniq.append(\"N\")\n        self.label_mode = label_mode\n        self.idx_to_chord = uniq\n        self.chord_to_idx = {c: i for i, c in enumerate(uniq)}\n\n    def encode(self, label: str) -> int:\n        return self.chord_to_idx.get(label, self.chord_to_idx[\"N\"])\n\n    def decode(self, idx: int) -> str:\n        return self.idx_to_chord[idx]\n\n    @property\n    def size(self) -> int:\n        return len(self.idx_to_chord)\n\n\nclass FixedChordVocab:\n    def __init__(self):\n        self.label_mode = \"quality27\"\n        self.idx_to_chord = TARGET_CLASSES\n        self.chord_to_idx = {c: i for i, c in enumerate(self.idx_to_chord)}\n\n    def encode(self, label: str) -> int:\n        return self.chord_to_idx[label]\n\n    def decode(self, idx: int) -> str:\n        return self.idx_to_chord[idx]\n\n    @property\n    def size(self):\n        return len(self.idx_to_chord)\n\n\nROOT_TO_CANONICAL = {\n    \"C\": \"C\",\n    \"B#\": \"C\",\n    \"C#\": \"C#\",\n    \"DB\": \"C#\",\n    \"D\": \"D\",\n    \"D#\": \"D#\",\n    \"EB\": \"D#\",\n    \"E\": \"E\",\n    \"FB\": \"E\",\n    \"E#\": \"F\",\n    \"F\": \"F\",\n    \"F#\": \"F#\",\n    \"GB\": \"F#\",\n    \"G\": \"G\",\n    \"G#\": \"G#\",\n    \"AB\": \"G#\",\n    \"A\": \"A\",\n    \"A#\": \"A#\",\n    \"BB\": \"A#\",\n    \"B\": \"B\",\n    \"CB\": \"B\",\n}\n\nCANONICAL_ROOTS = [\n    \"C\",\n    \"C#\",\n    \"D\",\n    \"D#\",\n    \"E\",\n    \"F\",\n    \"F#\",\n    \"G\",\n    \"G#\",\n    \"A\",\n    \"A#\",\n    \"B\",\n]\nFULL_CHORD_QUALITIES = [q for q in TARGET_CLASSES if q not in {\"Others\", \"N\"}]\nTRIAD_TYPES = [\"maj\", \"min\", \"dim\", \"aug\", \"sus4\", \"sus2\"]\nSTRUCTURED_COMPONENT_NAMES = [\n    \"root_triad\",\n    \"bass\",\n    \"seventh\",\n    \"ninth\",\n    \"eleventh\",\n    \"thirteenth\",\n]\nSTRUCTURED_COMPONENT_LABELS = {\n    \"root_triad\": [\"N\"] + [\n        f\"{root}:{triad}\"\n        for root in CANONICAL_ROOTS\n        for triad in TRIAD_TYPES\n    ],\n    \"bass\": [\"N\", \"root\", \"/2\", \"/b3\", \"/3\", \"/4\", \"/5\", \"/6\", \"/b7\"],\n    \"seventh\": [\"N\", \"none\", \"+7\", \"+b7\", \"+bb7\"],\n    \"ninth\": [\"N\", \"none\", \"+9\", \"+#9\", \"+b9\"],\n    \"eleventh\": [\"N\", \"none\", \"+11\", \"+#11\"],\n    \"thirteenth\": [\"N\", \"none\", \"+13\", \"+b13\"],\n}\n\n\ndef canonicalize_root(root: str) -> str | None:\n    return ROOT_TO_CANONICAL.get(root.strip().upper())\n\n\ndef normalize_quality(quality: str | None) -> str:\n    if quality is None:\n        return \"maj\"\n\n    quality = str(quality).strip().replace(\" \", \"\")\n    quality = quality.replace(\"*\", \"\")\n    if quality == \"\":\n        return \"maj\"\n    if quality in {\"1\", \"1/1\"}:\n        return \"maj\"\n    if quality.startswith(\"5\"):\n        return \"maj\"\n    if quality in TARGET_CLASSES and quality not in {\"N\", \"Others\"}:\n        return quality\n\n    base = re.sub(r\"\\([^)]*\\)\", \"\", quality)\n    if \"/\" in base:\n        base_no_bass = base.split(\"/\", 1)[0]\n    else:\n        base_no_bass = base\n\n    if quality.startswith(\"maj(9)\") or quality.startswith(\"maj9\"):\n        return \"maj9\"\n    if quality.startswith(\"min(9)\") or quality.startswith(\"min9\"):\n        return \"min9\"\n    if quality.startswith(\"7(\") or quality.startswith(\"7/\"):\n        return \"9\" if \"9\" in quality else \"7\"\n    if quality.startswith(\"maj7\"):\n        return \"maj7\"\n    if quality.startswith(\"min7\"):\n        return \"min7\"\n    if quality.startswith(\"sus4(b7\") or quality.startswith(\"sus4/b7\"):\n        return \"sus4(b7)\"\n    if quality.startswith(\"sus4\"):\n        return \"sus4\"\n    if quality.startswith(\"sus2\"):\n        return \"sus2\"\n    if quality.startswith(\"dim7\"):\n        return \"dim7\"\n    if quality.startswith(\"hdim7\"):\n        return \"hdim7\"\n    if quality.startswith(\"dim/b7\"):\n        return \"dim7\"\n    if quality.startswith(\"dim\"):\n        return \"dim\"\n    if quality.startswith(\"aug\"):\n        return \"aug\"\n    if quality.startswith(\"min11\"):\n        return \"min9\"\n    if quality.startswith(\"min6\"):\n        return \"min\"\n    if quality.startswith(\"maj6\"):\n        return \"maj\"\n    if quality.startswith(\"maj\"):\n        mapped_bass = _map_bass_marker(base)\n        return f\"maj{mapped_bass}\" if mapped_bass else \"maj\"\n    if quality.startswith(\"min\"):\n        mapped_bass = _map_bass_marker(base)\n        return f\"min{mapped_bass}\" if mapped_bass else \"min\"\n    if base_no_bass in TARGET_CLASSES and base_no_bass not in {\"N\", \"Others\"}:\n        return base_no_bass\n\n    if quality == \"maj6\":\n        return \"maj\"\n    if quality == \"min6\":\n        return \"min\"\n    if quality == \"maj6/9\":\n        return \"maj9\"\n    if quality == \"min6/9\":\n        return \"min9\"\n    if quality == \"minmaj7\":\n        return \"min7\"\n\n    return \"Others\"\n\n\ndef _map_bass_marker(quality: str) -> str | None:\n    if \"/\" not in quality:\n        return None\n    bass = quality.split(\"/\", 1)[1]\n    if bass in {\"2\", \"b3\", \"3\", \"5\", \"b7\"}:\n        return f\"/{bass}\"\n    return None\n\n\ndef chord_label_to_quality(label: str | None) -> str:\n    if label is None:\n        return \"Others\"\n\n    label = str(label).strip()\n\n    if label == \"\" or label.upper() == \"N\":\n        return \"N\"\n\n    if \":\" not in label:\n        if _looks_like_root_label(label):\n            return \"maj\"\n        return \"Others\"\n\n    _, quality = label.split(\":\", 1)\n    return normalize_quality(quality)\n\n\ndef chord_label_to_full_chord(label: str | None) -> str:\n    if label is None:\n        return \"N\"\n\n    label = str(label).strip()\n    if label == \"\" or label.upper() == \"N\":\n        return \"N\"\n\n    if \":\" not in label:\n        if \"/\" in label:\n            root_raw, bass_raw = label.split(\"/\", 1)\n            root = canonicalize_root(root_raw)\n            if root is not None and bass_raw in {\"2\", \"b3\", \"3\", \"5\", \"b7\"}:\n                return f\"{root}:maj/{bass_raw}\"\n        root = canonicalize_root(label)\n        return f\"{root}:maj\" if root is not None else \"N\"\n\n    root_raw, quality_raw = label.split(\":\", 1)\n    root = canonicalize_root(root_raw)\n    if root is None:\n        return \"N\"\n\n    quality = normalize_quality(quality_raw)\n    if quality in {\"N\", \"Others\"}:\n        return \"N\"\n    return f\"{root}:{quality}\"\n\n\ndef full_chord_to_components(label: str | None) -> List[str]:\n    label = chord_label_to_full_chord(label)\n    if label == \"N\":\n        return [\"N\", \"N\", \"N\", \"N\", \"N\", \"N\"]\n\n    root, quality = label.split(\":\", 1)\n\n    triad = \"maj\"\n    bass = \"root\"\n    seventh = \"none\"\n    ninth = \"none\"\n    eleventh = \"none\"\n    thirteenth = \"none\"\n\n    if quality.startswith(\"min\"):\n        triad = \"min\"\n    elif quality.startswith(\"dim\") or quality == \"hdim7\":\n        triad = \"dim\"\n    elif quality == \"aug\":\n        triad = \"aug\"\n    elif quality.startswith(\"sus4\"):\n        triad = \"sus4\"\n    elif quality.startswith(\"sus2\"):\n        triad = \"sus2\"\n\n    bass_markers = {\n        \"maj/2\": \"/2\",\n        \"min/2\": \"/2\",\n        \"min/b3\": \"/b3\",\n        \"maj/3\": \"/3\",\n        \"maj/5\": \"/5\",\n        \"min/5\": \"/5\",\n        \"maj/b7\": \"/b7\",\n        \"min/b7\": \"/b7\",\n    }\n    bass = bass_markers.get(quality, \"root\")\n\n    if quality == \"maj7\":\n        seventh = \"+7\"\n    elif quality in {\"7\", \"9\", \"11\", \"13\", \"min7\", \"min9\", \"sus4(b7)\", \"hdim7\"}:\n        seventh = \"+b7\"\n    elif quality == \"dim7\":\n        seventh = \"+bb7\"\n\n    if quality in {\"9\", \"maj9\", \"min9\", \"11\", \"13\"}:\n        ninth = \"+9\"\n    if quality in {\"11\", \"13\"}:\n        eleventh = \"+11\"\n    if quality == \"13\":\n        thirteenth = \"+13\"\n\n    return [\n        f\"{root}:{triad}\",\n        bass,\n        seventh,\n        ninth,\n        eleventh,\n        thirteenth,\n    ]\n\n\ndef build_structured_component_vocabs():\n    return {\n        name: {label: idx for idx, label in enumerate(labels)}\n        for name, labels in STRUCTURED_COMPONENT_LABELS.items()\n    }\n\n\ndef _looks_like_root_label(label: str) -> bool:\n    roots = {\n        \"C\", \"C#\", \"DB\", \"D\", \"D#\", \"EB\", \"E\", \"F\",\n        \"F#\", \"GB\", \"G\", \"G#\", \"AB\", \"A\", \"A#\", \"BB\", \"B\",\n    }\n    return label.strip().upper() in roots\n\ndef load_fold_split(fold_json_path: str):\n    with open(fold_json_path, \"r\", encoding=\"utf-8\") as f:\n        split_data = json.load(f)\n\n    train_ids = split_data[\"train\"]\n    val_ids = split_data[\"val\"]\n    test_ids = split_data[\"test\"]\n    return train_ids, val_ids, test_ids\n\n\ndef load_processed_npz(npz_path: str):\n    data = np.load(npz_path, allow_pickle=True)\n\n    cqt = data[\"cqt\"]          # [F, T]\n    labels = data[\"labels\"]    # [T]\n    sr = int(data[\"sr\"])\n    hop_length = int(data[\"hop_length\"])\n    frame_rate = float(data[\"frame_rate\"])\n    song_id = str(data[\"song_id\"])\n\n    # convert to [T, F]\n    x = cqt.T.astype(np.float32)\n\n    labels = labels.tolist()\n    labels = [str(lbl) for lbl in labels]\n\n    assert x.shape[0] == len(labels), f\"Mismatch: x has {x.shape[0]} frames but labels has {len(labels)}\"\n    \n    return {\n        \"x\": x,                      # [T, F]\n        \"labels\": labels,            # list length T\n        \"sr\": sr,\n        \"hop_length\": hop_length,\n        \"frame_rate\": frame_rate,\n        \"song_id\": song_id,\n    }\n\n\ndef make_chord_change_targets(chord_targets: np.ndarray) -> np.ndarray:\n    out = np.zeros_like(chord_targets, dtype=np.int64)\n    if len(chord_targets) > 1:\n        out[1:] = (chord_targets[1:] != chord_targets[:-1]).astype(np.int64)\n    out[0] = 0\n    return out\n\n\ndef slice_into_windows(\n    x: np.ndarray,             # [T, F]\n    chord_targets: np.ndarray, # [T]\n    chord_change_targets: np.ndarray, # [T]\n    chord_label_strings: List[str],\n    component_targets: np.ndarray | None,\n    n_steps: int,\n    stride: int,\n):\n    total_frames, feat_dim = x.shape\n    items = []\n\n    if total_frames <= n_steps:\n        pad_len = n_steps - total_frames\n\n        x_pad = np.pad(x, ((0, pad_len), (0, 0)), mode=\"constant\")\n        chord_pad = np.pad(chord_targets, (0, pad_len), mode=\"constant\", constant_values=0)\n        change_pad = np.pad(chord_change_targets, (0, pad_len), mode=\"constant\", constant_values=0)\n        if component_targets is not None:\n            component_pad = np.pad(\n                component_targets,\n                ((0, pad_len), (0, 0)),\n                mode=\"constant\",\n                constant_values=0,\n            )\n        else:\n            component_pad = None\n\n        mask = np.zeros(n_steps, dtype=np.float32)\n        mask[:total_frames] = 1.0\n\n        item = {\n            \"x\": x_pad.astype(np.float32),\n            \"chord_targets\": chord_pad.astype(np.int64),\n            \"chord_change_targets\": change_pad.astype(np.int64),\n            \"chord_label_strings\": chord_label_strings + [\"N\"] * pad_len,\n            \"mask\": mask,\n        }\n        if component_pad is not None:\n            item[\"component_targets\"] = component_pad.astype(np.int64)\n        items.append(item)\n        return items\n\n    for start in range(0, total_frames - n_steps + 1, stride):\n        end = start + n_steps\n        item = {\n            \"x\": x[start:end].astype(np.float32),\n            \"chord_targets\": chord_targets[start:end].astype(np.int64),\n            \"chord_change_targets\": chord_change_targets[start:end].astype(np.int64),\n            \"chord_label_strings\": chord_label_strings[start:end],\n            \"mask\": np.ones(n_steps, dtype=np.float32),\n        }\n        if component_targets is not None:\n            item[\"component_targets\"] = component_targets[start:end].astype(np.int64)\n        items.append(item)\n\n    if (total_frames - n_steps) % stride != 0:\n        start = total_frames - n_steps\n        end = total_frames\n        item = {\n            \"x\": x[start:end].astype(np.float32),\n            \"chord_targets\": chord_targets[start:end].astype(np.int64),\n            \"chord_change_targets\": chord_change_targets[start:end].astype(np.int64),\n            \"chord_label_strings\": chord_label_strings[start:end],\n            \"mask\": np.ones(n_steps, dtype=np.float32),\n        }\n        if component_targets is not None:\n            item[\"component_targets\"] = component_targets[start:end].astype(np.int64)\n        items.append(item)\n\n    return items\n\n\ndef make_song_item(\n    x: np.ndarray,\n    chord_targets: np.ndarray,\n    chord_change_targets: np.ndarray,\n    chord_label_strings: List[str],\n    component_targets: np.ndarray | None,\n):\n    item = {\n        \"x\": x.astype(np.float32),\n        \"chord_targets\": chord_targets.astype(np.int64),\n        \"chord_change_targets\": chord_change_targets.astype(np.int64),\n        \"chord_label_strings\": chord_label_strings,\n    }\n    if component_targets is not None:\n        item[\"component_targets\"] = component_targets.astype(np.int64)\n    return item\n\n\nclass ProcessedChordDataset(Dataset):\n    def __init__(self, items: List[Dict], augment: bool = False, cfg: ProcessedChordConfig | None = None):\n        self.items = items\n        self.augment = augment\n        self.cfg = cfg\n\n    def __len__(self):\n        return len(self.items)\n\n    def _augment_x(self, x: torch.Tensor) -> torch.Tensor:\n        if self.cfg is None:\n            return x\n\n        if self.cfg.use_signal_decay:\n            x = self._apply_signal_decay(x)\n\n        if self.cfg.noise_std > 0:\n            x = x + torch.randn_like(x) * self.cfg.noise_std\n\n        if self.cfg.gain_min > 0 and self.cfg.gain_max > 0:\n            gain = random.uniform(self.cfg.gain_min, self.cfg.gain_max)\n            x = x * gain\n\n        T, F = x.shape\n\n        if self.cfg.time_mask_width > 0 and T > 1:\n            width = random.randint(0, min(self.cfg.time_mask_width, T))\n            if width > 0:\n                start = random.randint(0, T - width)\n                x[start:start + width, :] = 0.0\n\n        if self.cfg.freq_mask_width > 0 and F > 1:\n            width = random.randint(0, min(self.cfg.freq_mask_width, F))\n            if width > 0:\n                start = random.randint(0, F - width)\n                x[:, start:start + width] = 0.0\n\n        if self.cfg.pitch_shift_bins > 0 and F > 1:\n            shift = random.randint(-self.cfg.pitch_shift_bins, self.cfg.pitch_shift_bins)\n            if shift != 0:\n                x = torch.roll(x, shifts=shift, dims=1)\n\n        if self.cfg.pitch_shift_semitones > 0 and F > 1:\n            semitones = random.randint(-self.cfg.pitch_shift_semitones, self.cfg.pitch_shift_semitones)\n            shift_bins = semitones * 3\n            if shift_bins != 0:\n                x = torch.roll(x, shifts=shift_bins, dims=1)\n\n        return x\n\n    def _apply_signal_decay(self, x: torch.Tensor) -> torch.Tensor:\n        T = x.shape[0]\n        if T <= 1:\n            return x\n\n        min_gain = min(self.cfg.signal_decay_min, self.cfg.signal_decay_max)\n        max_gain = max(self.cfg.signal_decay_min, self.cfg.signal_decay_max)\n        end_gain = random.uniform(min_gain, max_gain)\n        envelope = torch.linspace(1.0, end_gain, steps=T, dtype=x.dtype, device=x.device).unsqueeze(1)\n        return x * envelope\n\n    def __getitem__(self, idx):\n        item = self.items[idx]\n        x = torch.tensor(item[\"x\"], dtype=torch.float32)\n        if self.augment:\n            x = self._augment_x(x)\n\n        sample = {\n            \"x\": x,\n            \"chord_targets\": torch.tensor(item[\"chord_targets\"], dtype=torch.long),\n            \"chord_change_targets\": torch.tensor(item[\"chord_change_targets\"], dtype=torch.long),\n            \"mask\": torch.tensor(item[\"mask\"], dtype=torch.float32),\n        }\n        if \"component_targets\" in item:\n            sample[\"component_targets\"] = torch.tensor(item[\"component_targets\"], dtype=torch.long)\n        return sample\n\n\nclass DistributedEvalSampler(Sampler[int]):\n    def __init__(self, dataset: Dataset, num_replicas: int, rank: int):\n        self.dataset = dataset\n        self.num_replicas = num_replicas\n        self.rank = rank\n\n    def __iter__(self):\n        return iter(range(self.rank, len(self.dataset), self.num_replicas))\n\n    def __len__(self):\n        remaining = len(self.dataset) - self.rank\n        if remaining <= 0:\n            return 0\n        return (remaining + self.num_replicas - 1) // self.num_replicas\n\n\nclass RandomSongSegmentDataset(Dataset):\n    def __init__(self, songs: List[Dict], cfg: ProcessedChordConfig, augment: bool = False):\n        self.songs = songs\n        self.cfg = cfg\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.songs)\n\n    def _slice_song(self, item: Dict):\n        x = item[\"x\"]\n        chord_targets = item[\"chord_targets\"]\n        chord_change_targets = item[\"chord_change_targets\"]\n        component_targets = item.get(\"component_targets\")\n        total_frames = x.shape[0]\n        n_steps = self.cfg.n_steps\n\n        if total_frames <= n_steps:\n            pad_len = n_steps - total_frames\n            x_out = np.pad(x, ((0, pad_len), (0, 0)), mode=\"constant\")\n            chord_out = np.pad(chord_targets, (0, pad_len), mode=\"constant\", constant_values=0)\n            change_out = np.pad(chord_change_targets, (0, pad_len), mode=\"constant\", constant_values=0)\n            if component_targets is not None:\n                component_out = np.pad(\n                    component_targets,\n                    ((0, pad_len), (0, 0)),\n                    mode=\"constant\",\n                    constant_values=0,\n                )\n            else:\n                component_out = None\n            mask = np.zeros(n_steps, dtype=np.float32)\n            mask[:total_frames] = 1.0\n            label_strings = item[\"chord_label_strings\"][:total_frames] + [\"N\"] * pad_len\n        else:\n            start = random.randint(0, total_frames - n_steps)\n            end = start + n_steps\n            x_out = x[start:end]\n            chord_out = chord_targets[start:end]\n            change_out = chord_change_targets[start:end]\n            component_out = component_targets[start:end] if component_targets is not None else None\n            mask = np.ones(n_steps, dtype=np.float32)\n            label_strings = item[\"chord_label_strings\"][start:end]\n\n        out = {\n            \"x\": x_out.astype(np.float32),\n            \"chord_targets\": chord_out.astype(np.int64),\n            \"chord_change_targets\": change_out.astype(np.int64),\n            \"chord_label_strings\": label_strings,\n            \"mask\": mask,\n        }\n        if component_out is not None:\n            out[\"component_targets\"] = component_out.astype(np.int64)\n        return out\n\n    def __getitem__(self, idx):\n        item = self._slice_song(self.songs[idx])\n        sample = ProcessedChordDataset([item], augment=self.augment, cfg=self.cfg)[0]\n        return sample\n\n\ndef label_to_target(label: str, label_mode: str) -> str:\n    if label_mode == \"quality27\":\n        return chord_label_to_quality(label)\n    if label_mode in {\"full_chord\", \"structured_full_chord\"}:\n        return chord_label_to_full_chord(label)\n    raise ValueError(f\"Unsupported label_mode: {label_mode}\")\n\n\ndef build_vocab_from_train_ids(root_dir: str, train_ids: List[str], label_mode: str) -> ChordVocab:\n    all_labels = []\n\n    for track_id in train_ids:\n        npz_path = os.path.join(root_dir, \"processed\", f\"{track_id}.npz\")\n        if not os.path.exists(npz_path):\n            continue\n\n        data = np.load(npz_path, allow_pickle=True)\n        labels = [str(x) for x in data[\"labels\"].tolist()]\n        all_labels.extend(label_to_target(lbl, label_mode) for lbl in labels)\n\n    return ChordVocab(all_labels, label_mode=label_mode)\n\n\ndef attach_structured_metadata(vocab: ChordVocab) -> ChordVocab:\n    component_vocabs = build_structured_component_vocabs()\n    component_ids = []\n    for chord in vocab.idx_to_chord:\n        components = full_chord_to_components(chord)\n        component_ids.append([\n            component_vocabs[name][component]\n            for name, component in zip(STRUCTURED_COMPONENT_NAMES, components)\n        ])\n\n    vocab.component_names = STRUCTURED_COMPONENT_NAMES\n    vocab.component_labels = STRUCTURED_COMPONENT_LABELS\n    vocab.component_to_idx = component_vocabs\n    vocab.chord_component_ids = np.array(component_ids, dtype=np.int64)\n    return vocab\n\n\ndef build_full_chord_vocab(label_mode: str = \"full_chord\") -> ChordVocab:\n    labels = [\n        f\"{root}:{quality}\"\n        for root in CANONICAL_ROOTS\n        for quality in FULL_CHORD_QUALITIES\n    ]\n    labels.append(\"N\")\n    vocab = ChordVocab(labels, label_mode=label_mode)\n    return attach_structured_metadata(vocab)\n\n\ndef encode_component_targets(labels: List[str], vocab: ChordVocab) -> np.ndarray:\n    rows = []\n    for label in labels:\n        components = full_chord_to_components(label)\n        rows.append([\n            vocab.component_to_idx[name][component]\n            for name, component in zip(vocab.component_names, components)\n        ])\n    return np.array(rows, dtype=np.int64)\n\n\ndef build_items_from_ids(\n    root_dir: str,\n    track_ids: List[str],\n    vocab: ChordVocab,\n    cfg: ProcessedChordConfig,\n    window_mode: str | None = None,\n):\n    items = []\n    mode = window_mode or cfg.window_mode\n\n    for track_id in track_ids:\n        npz_path = os.path.join(root_dir, \"processed\", f\"{track_id}.npz\")\n        if not os.path.exists(npz_path):\n            continue\n\n        sample = load_processed_npz(npz_path)\n\n        x = sample[\"x\"]  # [T, F]\n        raw_label_strings = sample[\"labels\"]\n\n        target_labels = [label_to_target(lbl, cfg.label_mode) for lbl in raw_label_strings]\n        chord_targets = np.array([vocab.encode(lbl) for lbl in target_labels], dtype=np.int64)\n        chord_change_targets = make_chord_change_targets(chord_targets)\n        component_targets = None\n        if cfg.label_mode == \"structured_full_chord\":\n            component_targets = encode_component_targets(target_labels, vocab)\n\n        if mode == \"random_song\":\n            items.append(\n                make_song_item(\n                    x=x,\n                    chord_targets=chord_targets,\n                    chord_change_targets=chord_change_targets,\n                    chord_label_strings=target_labels,\n                    component_targets=component_targets,\n                )\n            )\n        elif mode == \"sliding\":\n            items.extend(\n                slice_into_windows(\n                    x=x,\n                    chord_targets=chord_targets,\n                    chord_change_targets=chord_change_targets,\n                    chord_label_strings=target_labels,\n                    component_targets=component_targets,\n                    n_steps=cfg.n_steps,\n                    stride=cfg.stride,\n                )\n            )\n        else:\n            raise ValueError(f\"Unsupported window_mode: {mode}\")\n\n    return items\n\n\ndef build_processed_loaders(cfg: ProcessedChordConfig, fold_json_path: str):\n    train_ids, val_ids, test_ids = load_fold_split(fold_json_path)\n\n    if cfg.label_mode == \"quality27\":\n        vocab = FixedChordVocab()\n    elif cfg.label_mode in {\"full_chord\", \"structured_full_chord\"}:\n        vocab = build_full_chord_vocab(label_mode=cfg.label_mode)\n    else:\n        raise ValueError(f\"Unsupported label_mode: {cfg.label_mode}\")\n\n    train_items = build_items_from_ids(cfg.root_dir, train_ids, vocab, cfg, window_mode=cfg.window_mode)\n    val_items = build_items_from_ids(cfg.root_dir, val_ids, vocab, cfg, window_mode=\"sliding\")\n    test_items = build_items_from_ids(cfg.root_dir, test_ids, vocab, cfg, window_mode=\"sliding\")\n\n    if cfg.window_mode == \"random_song\":\n        train_dataset = RandomSongSegmentDataset(train_items, cfg=cfg, augment=cfg.augment_train)\n        val_dataset = ProcessedChordDataset(val_items, augment=False, cfg=cfg)\n        test_dataset = ProcessedChordDataset(test_items, augment=False, cfg=cfg)\n    else:\n        train_dataset = ProcessedChordDataset(train_items, augment=cfg.augment_train, cfg=cfg)\n        val_dataset = ProcessedChordDataset(val_items, augment=False, cfg=cfg)\n        test_dataset = ProcessedChordDataset(test_items, augment=False, cfg=cfg)\n\n    train_sampler = None\n    val_sampler = None\n    test_sampler = None\n    if cfg.distributed:\n        train_sampler = DistributedSampler(\n            train_dataset,\n            num_replicas=cfg.world_size,\n            rank=cfg.rank,\n            shuffle=True,\n            drop_last=False,\n        )\n        val_sampler = DistributedEvalSampler(\n            val_dataset,\n            num_replicas=cfg.world_size,\n            rank=cfg.rank,\n        )\n        test_sampler = DistributedEvalSampler(\n            test_dataset,\n            num_replicas=cfg.world_size,\n            rank=cfg.rank,\n        )\n\n    loader_kwargs = {\n        \"batch_size\": cfg.batch_size,\n        \"num_workers\": cfg.num_workers,\n        \"pin_memory\": torch.cuda.is_available(),\n    }\n    if cfg.num_workers > 0:\n        loader_kwargs[\"persistent_workers\"] = True\n\n    train_loader = DataLoader(\n        train_dataset,\n        shuffle=train_sampler is None,\n        sampler=train_sampler,\n        **loader_kwargs,\n    )\n    val_loader = DataLoader(\n        val_dataset,\n        shuffle=False,\n        sampler=val_sampler,\n        **loader_kwargs,\n    )\n    test_loader = DataLoader(\n        test_dataset,\n        shuffle=False,\n        sampler=test_sampler,\n        **loader_kwargs,\n    )\n\n    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader, vocab\n",
    "evaluation_utils.py": "from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Callable, Dict, Iterable, List, Optional, Sequence\n\nimport numpy as np\nimport torch\nfrom tqdm.auto import tqdm\n\nfrom dataset import (\n    build_full_chord_vocab,\n    load_fold_split,\n    load_processed_npz,\n)\nfrom models import HyperParameters, build_model\n\n\n@dataclass\nclass EvalCheckpointConfig:\n    root_dir: str\n    checkpoint_path: str\n    model_type: str\n    label_mode: str\n    fold_id: int = 0\n    split: str = \"test\"\n    embed_size: int = 256\n    n_layers: int = 4\n    n_heads: int = 8\n    dropout: float = 0.3\n    n_steps: int = 128\n    stride: int = 64\n    device: str = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n\n\ndef _normalize_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:\n    normalized = {}\n    for key, value in state_dict.items():\n        if key.startswith(\"module.\"):\n            normalized[key[len(\"module.\"):]] = value\n        else:\n            normalized[key] = value\n    return normalized\n\n\ndef _first_processed_npz(root_dir: str) -> Path:\n    processed_dir = Path(root_dir) / \"processed\"\n    try:\n        return next(processed_dir.glob(\"*.npz\"))\n    except StopIteration as exc:\n        raise FileNotFoundError(f\"No processed npz files found in {processed_dir}\") from exc\n\n\ndef infer_input_dim(root_dir: str) -> int:\n    sample = load_processed_npz(str(_first_processed_npz(root_dir)))\n    return int(sample[\"x\"].shape[-1])\n\n\ndef load_eval_model(cfg: EvalCheckpointConfig):\n    if cfg.label_mode not in {\"full_chord\", \"structured_full_chord\"}:\n        raise ValueError(\n            \"Notebook export/eval expects full chord labels. \"\n            \"Use label_mode='full_chord' or 'structured_full_chord'.\"\n        )\n\n    vocab = build_full_chord_vocab(label_mode=cfg.label_mode)\n    hp = HyperParameters(\n        n_steps=cfg.n_steps,\n        input_embed_size=cfg.embed_size,\n        n_layers=cfg.n_layers,\n        n_heads=cfg.n_heads,\n    )\n    device = torch.device(cfg.device)\n    model = build_model(\n        model_type=cfg.model_type,\n        input_dim=infer_input_dim(cfg.root_dir),\n        vocab=vocab,\n        hyperparameters=hp,\n        dropout_rate=cfg.dropout,\n    ).to(device)\n\n    checkpoint = torch.load(cfg.checkpoint_path, map_location=device)\n    if isinstance(checkpoint, dict) and \"state_dict\" in checkpoint:\n        checkpoint = checkpoint[\"state_dict\"]\n    if not isinstance(checkpoint, dict):\n        raise ValueError(f\"Unsupported checkpoint format in {cfg.checkpoint_path}\")\n\n    model.load_state_dict(_normalize_state_dict(checkpoint), strict=True)\n    model.eval()\n    return model, vocab, device\n\n\ndef resolve_song_ids(root_dir: str, fold_id: int, split: str) -> List[str]:\n    split = split.lower()\n    if split == \"all\":\n        return sorted(p.stem for p in (Path(root_dir) / \"processed\").glob(\"*.npz\"))\n\n    fold_json = Path(root_dir) / \"splits\" / f\"fold_{fold_id}.json\"\n    train_ids, val_ids, test_ids = load_fold_split(str(fold_json))\n    if split == \"train\":\n        return list(train_ids)\n    if split == \"val\":\n        return list(val_ids)\n    if split == \"test\":\n        return list(test_ids)\n    raise ValueError(f\"Unsupported split: {split}\")\n\n\ndef make_window_starts(total_frames: int, n_steps: int, stride: int) -> List[int]:\n    if total_frames <= n_steps:\n        return [0]\n    starts = list(range(0, total_frames - n_steps + 1, stride))\n    last_start = total_frames - n_steps\n    if starts[-1] != last_start:\n        starts.append(last_start)\n    return starts\n\n\ndef apply_signal_decay(x: np.ndarray, final_gain: float) -> np.ndarray:\n    envelope = np.linspace(1.0, float(final_gain), num=x.shape[0], dtype=np.float32)[:, None]\n    return x * envelope\n\n\ndef apply_feature_noise_std(x: np.ndarray, noise_std: float, seed: int) -> np.ndarray:\n    if noise_std <= 0:\n        return x\n    rng = np.random.default_rng(seed)\n    noise = rng.normal(loc=0.0, scale=float(noise_std), size=x.shape).astype(np.float32)\n    return x + noise\n\n\ndef apply_feature_noise_snr(x: np.ndarray, snr_db: float, seed: int) -> np.ndarray:\n    if not np.isfinite(snr_db):\n        return x\n    signal_power = float(np.mean(np.square(x.astype(np.float64))))\n    if signal_power <= 0:\n        return x\n    noise_power = signal_power / (10.0 ** (float(snr_db) / 10.0))\n    rng = np.random.default_rng(seed)\n    noise = rng.normal(loc=0.0, scale=np.sqrt(noise_power), size=x.shape).astype(np.float32)\n    return x + noise\n\n\ndef _batched(iterable: Sequence[int], batch_size: int) -> Iterable[Sequence[int]]:\n    for start in range(0, len(iterable), batch_size):\n        yield iterable[start:start + batch_size]\n\n\n@torch.no_grad()\ndef predict_song_labels(\n    model,\n    vocab,\n    x: np.ndarray,\n    device: torch.device,\n    n_steps: int,\n    stride: int,\n    batch_size: int = 32,\n) -> List[str]:\n    total_frames, feat_dim = x.shape\n    starts = make_window_starts(total_frames, n_steps, stride)\n    chord_logits_sum = None\n    frame_counts = np.zeros(total_frames, dtype=np.float32)\n\n    for batch_starts in _batched(starts, batch_size):\n        batch_x = np.zeros((len(batch_starts), n_steps, feat_dim), dtype=np.float32)\n        batch_mask = np.zeros((len(batch_starts), n_steps), dtype=np.float32)\n        valid_lengths: List[int] = []\n\n        for batch_idx, start in enumerate(batch_starts):\n            end = min(start + n_steps, total_frames)\n            valid_len = end - start\n            batch_x[batch_idx, :valid_len] = x[start:end]\n            batch_mask[batch_idx, :valid_len] = 1.0\n            valid_lengths.append(valid_len)\n\n        outputs = model(\n            x=torch.from_numpy(batch_x).to(device),\n            source_mask=torch.from_numpy(batch_mask).to(device),\n            target_mask=torch.from_numpy(batch_mask).to(device),\n            slope=1.0,\n        )\n        batch_logits = outputs[\"chord_logits\"].detach().cpu().numpy().astype(np.float32)\n        if chord_logits_sum is None:\n            chord_logits_sum = np.zeros((total_frames, batch_logits.shape[-1]), dtype=np.float32)\n\n        for batch_idx, start in enumerate(batch_starts):\n            valid_len = valid_lengths[batch_idx]\n            chord_logits_sum[start:start + valid_len] += batch_logits[batch_idx, :valid_len]\n            frame_counts[start:start + valid_len] += 1.0\n\n    if chord_logits_sum is None:\n        raise ValueError(\"No logits were produced during song prediction\")\n\n    frame_counts = np.maximum(frame_counts, 1.0)\n    chord_logits = chord_logits_sum / frame_counts[:, None]\n    pred_ids = chord_logits.argmax(axis=-1)\n    return [vocab.decode(int(idx)) for idx in pred_ids]\n\n\ndef labels_to_lab_lines(labels: Sequence[str], frame_rate: float) -> List[str]:\n    if not labels:\n        return []\n\n    lines: List[str] = []\n    start_idx = 0\n    current_label = labels[0]\n    for idx in range(1, len(labels)):\n        if labels[idx] != current_label:\n            start_t = start_idx / frame_rate\n            end_t = idx / frame_rate\n            lines.append(f\"{start_t:.6f}\\t{end_t:.6f}\\t{current_label}\")\n            start_idx = idx\n            current_label = labels[idx]\n\n    start_t = start_idx / frame_rate\n    end_t = len(labels) / frame_rate\n    lines.append(f\"{start_t:.6f}\\t{end_t:.6f}\\t{current_label}\")\n    return lines\n\n\ndef save_lab(labels: Sequence[str], frame_rate: float, output_path: str) -> None:\n    output = Path(output_path)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(\"\\n\".join(labels_to_lab_lines(labels, frame_rate)) + \"\\n\", encoding=\"utf-8\")\n\n\ndef export_predictions(\n    cfg: EvalCheckpointConfig,\n    output_dir: str,\n    corruption_fn: Optional[Callable[[np.ndarray, str], np.ndarray]] = None,\n    batch_size: int = 32,\n    overwrite: bool = False,\n    limit: Optional[int] = None,\n) -> Dict[str, object]:\n    model, vocab, device = load_eval_model(cfg)\n    song_ids = resolve_song_ids(cfg.root_dir, cfg.fold_id, cfg.split)\n    if limit is not None:\n        song_ids = song_ids[:limit]\n\n    output_dir_path = Path(output_dir)\n    output_dir_path.mkdir(parents=True, exist_ok=True)\n\n    exported = 0\n    skipped = 0\n    for song_id in tqdm(song_ids, desc=output_dir_path.name or \"predict\", leave=False):\n        out_path = output_dir_path / f\"{song_id}.lab\"\n        if out_path.exists() and not overwrite:\n            skipped += 1\n            continue\n\n        sample = load_processed_npz(str(Path(cfg.root_dir) / \"processed\" / f\"{song_id}.npz\"))\n        x = sample[\"x\"]\n        if corruption_fn is not None:\n            x = corruption_fn(x, song_id)\n        labels = predict_song_labels(\n            model=model,\n            vocab=vocab,\n            x=x,\n            device=device,\n            n_steps=cfg.n_steps,\n            stride=cfg.stride,\n            batch_size=batch_size,\n        )\n        save_lab(labels, float(sample[\"frame_rate\"]), str(out_path))\n        exported += 1\n\n    return {\n        \"exported\": exported,\n        \"skipped\": skipped,\n        \"total\": len(song_ids),\n        \"output_dir\": str(output_dir_path),\n        \"vocab_labels\": list(vocab.idx_to_chord),\n    }\n\n\ndef _load_mir_eval():\n    import mir_eval\n    return mir_eval\n\n\ndef evaluate_mir_eval_metrics(est_dir: str, ref_dir: str) -> Dict[str, float]:\n    mir_eval = _load_mir_eval()\n    metric_fns = [\n        (\"root\", mir_eval.chord.root),\n        (\"thirds\", mir_eval.chord.thirds),\n        (\"majmin\", mir_eval.chord.majmin),\n        (\"triads\", mir_eval.chord.triads),\n        (\"sevenths\", mir_eval.chord.sevenths),\n        (\"tetrads\", mir_eval.chord.tetrads),\n        (\"mirex\", mir_eval.chord.mirex),\n    ]\n\n    per_song = []\n    for est_path in tqdm(sorted(Path(est_dir).glob(\"*.lab\")), desc=Path(est_dir).name or \"mir_eval\", leave=False):\n        ref_path = Path(ref_dir) / est_path.name\n        if not ref_path.exists():\n            continue\n        try:\n            ref_intervals, ref_labels = mir_eval.io.load_labeled_intervals(str(ref_path))\n            est_intervals, est_labels = mir_eval.io.load_labeled_intervals(str(est_path))\n            est_intervals, est_labels = mir_eval.util.adjust_intervals(\n                est_intervals,\n                est_labels,\n                float(ref_intervals.min()),\n                float(ref_intervals.max()),\n                mir_eval.chord.NO_CHORD,\n                mir_eval.chord.NO_CHORD,\n            )\n            intervals, merged_ref, merged_est = mir_eval.util.merge_labeled_intervals(\n                ref_intervals,\n                ref_labels,\n                est_intervals,\n                est_labels,\n            )\n            durations = mir_eval.util.intervals_to_durations(intervals)\n        except Exception:\n            continue\n\n        scores = {}\n        for name, fn in metric_fns:\n            comparisons = fn(merged_ref, merged_est)\n            scores[name] = float(mir_eval.chord.weighted_accuracy(comparisons, durations))\n        per_song.append(scores)\n\n    if not per_song:\n        return {name: float(\"nan\") for name, _ in metric_fns}\n\n    return {\n        name: float(np.mean([song[name] for song in per_song]))\n        for name, _ in metric_fns\n    }\n\n\ndef build_mir_eval_vocab_keys(vocab_labels: Sequence[str]):\n    mir_eval = _load_mir_eval()\n\n    def chord_key(label: str):\n        if label is None or label == \"\" or label == \"X\":\n            return None\n        if label == \"N\":\n            return (\"N\",)\n        try:\n            root, intervals, bass = mir_eval.chord.encode(label, reduce_extended_chords=False)\n        except mir_eval.chord.InvalidChordException:\n            return None\n        if root < 0:\n            return (\"N\",)\n        return (int(root), tuple(int(x) for x in intervals), int(bass))\n\n    vocab_keys = set()\n    for label in vocab_labels:\n        key = chord_key(label)\n        if key is not None:\n            vocab_keys.add(key)\n    return chord_key, vocab_keys\n\n\ndef rasterize_labels(intervals, labels, n_frames: int, fps: float) -> np.ndarray:\n    out = np.full(n_frames, \"N\", dtype=object)\n    for (start, end), label in zip(intervals, labels):\n        frame_start = max(0, int(round(float(start) * fps)))\n        frame_end = min(n_frames, int(round(float(end) * fps)))\n        if frame_end > frame_start:\n            out[frame_start:frame_end] = label\n    return out\n\n\ndef evaluate_large_vocabulary(\n    est_dir: str,\n    ref_dir: str,\n    vocab_labels: Sequence[str],\n    fps: float = 22050.0 / 512.0,\n) -> Dict[str, float]:\n    mir_eval = _load_mir_eval()\n    chord_key, vocab_keys = build_mir_eval_vocab_keys(vocab_labels)\n\n    total = 0\n    correct = 0\n    cls_correct = {key: 0 for key in vocab_keys}\n    cls_total = {key: 0 for key in vocab_keys}\n\n    for est_path in tqdm(sorted(Path(est_dir).glob(\"*.lab\")), desc=Path(est_dir).name or \"large_vocab\", leave=False):\n        ref_path = Path(ref_dir) / est_path.name\n        if not ref_path.exists():\n            continue\n        try:\n            ref_intervals, ref_labels = mir_eval.io.load_labeled_intervals(str(ref_path))\n            est_intervals, est_labels = mir_eval.io.load_labeled_intervals(str(est_path))\n        except Exception:\n            continue\n\n        end_time = min(float(ref_intervals[-1, 1]), float(est_intervals[-1, 1]))\n        if end_time <= 0:\n            continue\n\n        n_frames = int(end_time * fps)\n        ref_frames = rasterize_labels(ref_intervals, ref_labels, n_frames, fps)\n        est_frames = rasterize_labels(est_intervals, est_labels, n_frames, fps)\n\n        key_cache: Dict[str, object] = {}\n\n        def get_key(label: str):\n            if label not in key_cache:\n                key_cache[label] = chord_key(label)\n            return key_cache[label]\n\n        for ref_label, est_label in zip(ref_frames, est_frames):\n            ref_key = get_key(ref_label)\n            if ref_key is None or ref_key not in vocab_keys:\n                continue\n            est_key = get_key(est_label)\n            total += 1\n            cls_total[ref_key] += 1\n            if est_key == ref_key:\n                correct += 1\n                cls_correct[ref_key] += 1\n\n    seen = [key for key in vocab_keys if cls_total[key] > 0]\n    frame_acc = (correct / total) if total > 0 else float(\"nan\")\n    class_acc = (\n        float(np.mean([cls_correct[key] / cls_total[key] for key in seen]))\n        if seen else float(\"nan\")\n    )\n    return {\n        \"frame_acc\": frame_acc,\n        \"class_acc\": class_acc,\n        \"frames\": int(total),\n        \"classes_seen\": int(len(seen)),\n        \"vocab_key_count\": int(len(vocab_keys)),\n    }\n",
    "models/__init__.py": "from .btc import BTCChordModel, StructuredBTCChordModel\nfrom .htv2 import HTv2ChordModel, HyperParameters, StructuredHTv2ChordModel\n\n\ndef build_model(\n    model_type: str,\n    input_dim: int,\n    vocab,\n    hyperparameters: HyperParameters,\n    dropout_rate: float,\n):\n    if getattr(vocab, \"label_mode\", \"\") == \"structured_full_chord\":\n        component_sizes = {\n            name: len(vocab.component_labels[name])\n            for name in vocab.component_names\n        }\n        if model_type == \"htv2\":\n            return StructuredHTv2ChordModel(\n                input_dim=input_dim,\n                component_sizes=component_sizes,\n                chord_component_ids=vocab.chord_component_ids,\n                hyperparameters=hyperparameters,\n                dropout_rate=dropout_rate,\n            )\n        if model_type == \"btc\":\n            return StructuredBTCChordModel(\n                input_dim=input_dim,\n                component_sizes=component_sizes,\n                chord_component_ids=vocab.chord_component_ids,\n                hyperparameters=hyperparameters,\n                dropout_rate=dropout_rate,\n            )\n    else:\n        if model_type == \"htv2\":\n            return HTv2ChordModel(\n                input_dim=input_dim,\n                n_chords=vocab.size,\n                hyperparameters=hyperparameters,\n                dropout_rate=dropout_rate,\n            )\n        if model_type == \"btc\":\n            return BTCChordModel(\n                input_dim=input_dim,\n                n_chords=vocab.size,\n                hyperparameters=hyperparameters,\n                dropout_rate=dropout_rate,\n            )\n\n    raise ValueError(f\"Unsupported model_type: {model_type}\")\n",
    "models/htv2.py": "import math\nfrom dataclasses import dataclass\nfrom typing import Optional, Tuple, List, Dict\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n# =========================================================\n# Hyperparameters\n# =========================================================\n\n@dataclass\nclass HyperParameters:\n    n_steps: int\n    input_embed_size: int\n    n_layers: int\n    n_heads: int\n\n\n# =========================================================\n# Positional encodings\n# =========================================================\n\ndef get_absolute_position_encoding(\n    length: int,\n    hidden_size: int,\n    min_timescale: float = 1.0,\n    max_timescale: float = 1e4,\n    start_index: int = 0,\n    device: Optional[torch.device] = None,\n    dtype: torch.dtype = torch.float32,\n) -> torch.Tensor:\n    position = torch.arange(length, device=device, dtype=dtype) + start_index\n    num_timescales = hidden_size // 2\n\n    if num_timescales > 1:\n        log_timescale_increment = math.log(max_timescale / min_timescale) / (num_timescales - 1)\n    else:\n        log_timescale_increment = 0.0\n\n    inv_timescales = min_timescale * torch.exp(\n        torch.arange(num_timescales, device=device, dtype=dtype) * (-log_timescale_increment)\n    )\n    scaled_time = position[:, None] * inv_timescales[None, :]\n    signal = torch.cat([torch.sin(scaled_time), torch.cos(scaled_time)], dim=1)\n\n    if hidden_size % 2 == 1:\n        signal = F.pad(signal, (0, 1))\n\n    return signal.unsqueeze(0)  # [1, L, C]\n\n\ndef get_relative_position_encoding(\n    length_q: int,\n    length_k: int,\n    n_units: int,\n    max_dist: int,\n    device: torch.device,\n    dtype: torch.dtype,\n) -> torch.Tensor:\n    range_vec_k = torch.arange(length_k, device=device)\n    if length_q == length_k:\n        range_vec_q = range_vec_k\n    else:\n        range_vec_q = range_vec_k[-length_q:]\n\n    distance_mat = range_vec_k[None, :] - range_vec_q[:, None]\n    distance_mat_clipped = torch.clamp(distance_mat, -max_dist, max_dist)\n    final_mat = distance_mat_clipped + max_dist\n\n    vocab_size = max_dist * 2 + 1\n    embeddings_table = get_absolute_position_encoding(\n        vocab_size,\n        n_units,\n        device=device,\n        dtype=dtype,\n    ).squeeze(0)\n\n    embeddings = embeddings_table[final_mat]  # [T_q, T_k, C]\n    return embeddings\n\n\n# =========================================================\n# LayerNorm\n# =========================================================\n\nclass CustomLayerNorm(nn.Module):\n    def __init__(self, hidden_size: int, eps: float = 1e-6):\n        super().__init__()\n        self.gamma = nn.Parameter(torch.ones(hidden_size))\n        self.beta = nn.Parameter(torch.zeros(hidden_size))\n        self.eps = eps\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        mean = x.mean(dim=-1, keepdim=True)\n        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)\n        x_norm = (x - mean) * torch.rsqrt(var + self.eps)\n        return self.gamma * x_norm + self.beta\n\n\n# =========================================================\n# Straight-through binary round\n# =========================================================\n\nclass BinaryRoundSTE(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x):\n        return torch.round(x)\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        return grad_output\n\n\ndef binary_round(x: torch.Tensor, cast_to_int: bool = False) -> torch.Tensor:\n    y = BinaryRoundSTE.apply(x)\n    if cast_to_int:\n        return y.long()\n    return y\n\n\n# =========================================================\n# Feed-forward blocks\n# =========================================================\n\nclass FFN(nn.Module):\n    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout_rate: float):\n        super().__init__()\n        self.conv1 = nn.Conv1d(in_dim, hidden_dim, kernel_size=1)\n        self.conv2 = nn.Conv1d(hidden_dim, out_dim, kernel_size=1)\n        self.dropout = nn.Dropout(dropout_rate)\n        self.norm = CustomLayerNorm(out_dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = x\n        y = x.transpose(1, 2)\n        y = F.relu(self.conv1(y))\n        y = self.dropout(y)\n        y = self.conv2(y)\n        y = self.dropout(y)\n        y = y.transpose(1, 2)\n        y = y + residual\n        y = self.norm(y)\n        return y\n\n\nclass ConvFFN(nn.Module):\n    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout_rate: float):\n        super().__init__()\n        self.conv1 = nn.Conv1d(in_dim, hidden_dim, kernel_size=3, padding=1)\n        self.conv2 = nn.Conv1d(hidden_dim, out_dim, kernel_size=3, padding=1)\n        self.dropout = nn.Dropout(dropout_rate)\n        self.norm = CustomLayerNorm(out_dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        residual = x\n        y = x.transpose(1, 2)\n        y = F.relu(self.conv1(y))\n        y = self.dropout(y)\n        y = F.relu(self.conv2(y))\n        y = self.dropout(y)\n        y = y.transpose(1, 2)\n        y = y + residual\n        y = self.norm(y)\n        return y\n\n\n# =========================================================\n# Multi-head attention\n# =========================================================\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(\n        self,\n        n_units: int,\n        n_heads: int,\n        dropout_rate: float,\n        relative_position: bool = False,\n        max_dist: int = 4,\n        positional_attention: bool = False,\n    ):\n        super().__init__()\n        assert n_units % n_heads == 0, \"n_units must be divisible by n_heads\"\n\n        self.n_units = n_units\n        self.n_heads = n_heads\n        self.head_dim = n_units // n_heads\n        self.relative_position = relative_position\n        self.max_dist = max_dist\n        self.positional_attention = positional_attention\n\n        self.q_proj = nn.Linear(n_units, n_units)\n        self.k_proj = nn.Linear(n_units, n_units)\n        self.v_proj = nn.Linear(n_units, n_units)\n        self.o_proj = nn.Linear(n_units, n_units)\n\n        if relative_position:\n            self.pe_u = nn.Parameter(torch.zeros(n_units))\n            self.pe_v = nn.Parameter(torch.zeros(n_units))\n            self.rel_pe_proj = nn.Linear(n_units, n_units)\n\n        self.dropout = nn.Dropout(dropout_rate)\n        self.norm = CustomLayerNorm(n_units)\n\n    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:\n        N, T, C = x.shape\n        x = x.view(N, T, self.n_heads, self.head_dim)\n        return x.permute(0, 2, 1, 3)  # [N, H, T, D]\n\n    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:\n        N, H, T, D = x.shape\n        x = x.permute(0, 2, 1, 3).contiguous()\n        return x.view(N, T, H * D)\n\n    def forward(\n        self,\n        queries: torch.Tensor,\n        keys: torch.Tensor,\n        values: Optional[torch.Tensor] = None,\n        key_mask: Optional[torch.Tensor] = None,\n        attention_map: bool = False,\n    ):\n        if values is None:\n            values = keys\n\n        residual = values if self.positional_attention else queries\n\n        Q = self.dropout(self.q_proj(queries))\n        K = self.dropout(self.k_proj(keys))\n        V = self.dropout(self.v_proj(values))\n\n        N, T_q, _ = Q.shape\n        _, T_k, _ = K.shape\n\n        Qh = self._split_heads(Q)\n        Kh = self._split_heads(K)\n        Vh = self._split_heads(V)\n\n        if not self.relative_position:\n            scores = torch.matmul(Qh, Kh.transpose(-1, -2))\n        else:\n            q_u = Q + self.pe_u.view(1, 1, -1)\n            q_u = self._split_heads(q_u)\n            ac = torch.matmul(q_u, Kh.transpose(-1, -2))\n\n            rel_pe = get_relative_position_encoding(\n                length_q=T_q,\n                length_k=T_k,\n                n_units=self.n_units,\n                max_dist=self.max_dist,\n                device=Q.device,\n                dtype=Q.dtype,\n            )\n            rel_pe = self.dropout(self.rel_pe_proj(rel_pe))\n            rel_pe = rel_pe.view(T_q, T_k, self.n_heads, self.head_dim).permute(2, 0, 1, 3)\n\n            q_v = Q + self.pe_v.view(1, 1, -1)\n            q_v = self._split_heads(q_v)\n\n            bd = torch.einsum(\"nhtd,htkd->nhtk\", q_v, rel_pe)\n            scores = ac + bd\n\n        scores = scores / math.sqrt(self.head_dim)\n\n        if key_mask is not None:\n            key_mask = key_mask.bool()\n            mask = key_mask[:, None, None, :]\n            scores = scores.masked_fill(~mask, -1e9)\n\n        attn = F.softmax(scores, dim=-1)\n        out = torch.matmul(attn, Vh)\n\n        out = self._merge_heads(out)\n        out = self.dropout(self.o_proj(out))\n        out = out + residual\n        out = self.norm(out)\n\n        if attention_map:\n            attn_map = attn.permute(0, 2, 1, 3).contiguous().view(N, T_q, self.n_heads * T_k)\n            return out, attn_map\n\n        return out\n\n\n# =========================================================\n# Chord block compression\n# =========================================================\n\ndef chord_block_compression(\n    hidden_states: torch.Tensor,\n    chord_changes: torch.Tensor,\n    compression: str = \"mean\",\n) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n    if compression not in {\"mean\", \"sum\"}:\n        raise ValueError(\"compression must be 'mean' or 'sum'\")\n\n    N, T, C = hidden_states.shape\n    device = hidden_states.device\n    dtype = hidden_states.dtype\n\n    block_ids = torch.cumsum(chord_changes, dim=1)\n    change_at_start = (chord_changes[:, 0] == 1).long()\n    block_ids = block_ids - change_at_start[:, None]\n\n    num_blocks = block_ids.max(dim=1).values + 1\n    max_steps = int(num_blocks.max().item())\n\n    chord_blocks = []\n    for b in range(N):\n        h = hidden_states[b]\n        ids = block_ids[b]\n        nb = int(num_blocks[b].item())\n\n        blocks = []\n        for seg_id in range(nb):\n            seg = h[ids == seg_id]\n            if seg.numel() == 0:\n                blk = torch.zeros(C, device=device, dtype=dtype)\n            else:\n                blk = seg.mean(dim=0) if compression == \"mean\" else seg.sum(dim=0)\n            blocks.append(blk)\n\n        blocks = torch.stack(blocks, dim=0)\n        if nb < max_steps:\n            pad = torch.zeros(max_steps - nb, C, device=device, dtype=dtype)\n            blocks = torch.cat([blocks, pad], dim=0)\n\n        chord_blocks.append(blocks)\n\n    chord_blocks = torch.stack(chord_blocks, dim=0)\n    return chord_blocks, block_ids, num_blocks\n\n\ndef decode_compressed_sequences(\n    compressed_sequences: torch.Tensor,\n    block_ids: torch.Tensor,\n) -> torch.Tensor:\n    outputs = []\n    for b in range(block_ids.shape[0]):\n        outputs.append(compressed_sequences[b][block_ids[b]])\n    return torch.stack(outputs, dim=0)\n\n\n# =========================================================\n# Intra-block attention\n# =========================================================\n\nclass IntraBlockMHA(nn.Module):\n    def __init__(self, n_units: int, n_heads: int, dropout_rate: float):\n        super().__init__()\n        self.mha = MultiHeadAttention(\n            n_units=n_units,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            relative_position=True,\n            max_dist=3,\n        )\n        self.ffn = ConvFFN(\n            in_dim=n_units,\n            hidden_dim=n_units,\n            out_dim=n_units,\n            dropout_rate=dropout_rate,\n        )\n\n    def forward(self, inputs: torch.Tensor, n_blocks: int, mask: torch.Tensor) -> torch.Tensor:\n        N, T, C = inputs.shape\n        assert T % n_blocks == 0, \"T must be divisible by n_blocks\"\n\n        block_len = T // n_blocks\n        blocks = inputs.view(N, n_blocks, block_len, C).reshape(N * n_blocks, block_len, C)\n        block_mask = mask.view(N, n_blocks, block_len).reshape(N * n_blocks, block_len)\n\n        blocks = self.mha(blocks, blocks, key_mask=block_mask)\n        blocks = self.ffn(blocks)\n\n        blocks = blocks.view(N, n_blocks, block_len, C).reshape(N, T, C)\n        return blocks\n\n\n# =========================================================\n# Encoder / Decoder layers\n# =========================================================\n\nclass HTv2EncoderLayer(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, n_steps: int, dropout_rate: float):\n        super().__init__()\n        self.self_attn = MultiHeadAttention(\n            n_units=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            relative_position=True,\n            max_dist=n_steps - 1,\n        )\n        self.ffn = ConvFFN(d_model, d_model, d_model, dropout_rate)\n\n    def forward(self, x: torch.Tensor, source_mask: torch.Tensor) -> torch.Tensor:\n        x = self.self_attn(x, x, key_mask=source_mask)\n        x = self.ffn(x)\n        return x\n\n\nclass HTv2DecoderLayer(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, n_steps: int, dropout_rate: float):\n        super().__init__()\n        self.self_attn = MultiHeadAttention(\n            n_units=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            relative_position=True,\n            max_dist=n_steps - 1,\n        )\n        self.position_attn = MultiHeadAttention(\n            n_units=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            relative_position=True,\n            max_dist=n_steps - 1,\n            positional_attention=True,\n        )\n        self.enc_dec_attn = MultiHeadAttention(\n            n_units=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            relative_position=False,\n        )\n        self.ffn = ConvFFN(d_model, d_model, d_model, dropout_rate)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        enc: torch.Tensor,\n        dec_pe_batch: torch.Tensor,\n        target_mask: torch.Tensor,\n        source_mask: torch.Tensor,\n    ):\n        x, self_attn_map = self.self_attn(\n            queries=x,\n            keys=x,\n            key_mask=target_mask,\n            attention_map=True,\n        )\n\n        x = self.position_attn(\n            queries=dec_pe_batch,\n            keys=dec_pe_batch,\n            values=x,\n            key_mask=target_mask,\n        )\n\n        x, attn_map = self.enc_dec_attn(\n            queries=x,\n            keys=enc,\n            key_mask=source_mask,\n            attention_map=True,\n        )\n\n        x = self.ffn(x)\n        return x, self_attn_map, attn_map\n\n\n# =========================================================\n# HTv2 backbone\n# =========================================================\n\nclass HTv2(nn.Module):\n    def __init__(self, input_dim: int, hyperparameters: HyperParameters, dropout_rate: float):\n        super().__init__()\n        hp = hyperparameters\n        self.hp = hp\n        self.dropout = nn.Dropout(dropout_rate)\n\n        if hp.n_steps % 4 != 0:\n            raise ValueError(\"hyperparameters.n_steps must be divisible by 4\")\n\n        self.enc_input_proj = nn.Linear(input_dim, hp.input_embed_size)\n        self.enc_intra_block = IntraBlockMHA(hp.input_embed_size, hp.n_heads, dropout_rate)\n\n        self.enc_layer_weights = nn.Parameter(torch.zeros(hp.n_layers + 1))\n        self.encoder_layers = nn.ModuleList([\n            HTv2EncoderLayer(hp.input_embed_size, hp.n_heads, hp.n_steps, dropout_rate)\n            for _ in range(hp.n_layers)\n        ])\n\n        self.chord_change_head = nn.Linear(hp.input_embed_size, 1)\n\n        self.dec_input_proj = nn.Linear(input_dim, hp.input_embed_size)\n        self.dec_intra_block = IntraBlockMHA(hp.input_embed_size, hp.n_heads, dropout_rate)\n\n        self.dec_layer_weights = nn.Parameter(torch.zeros(hp.n_layers + 1))\n        self.decoder_layers = nn.ModuleList([\n            HTv2DecoderLayer(hp.input_embed_size, hp.n_heads, hp.n_steps, dropout_rate)\n            for _ in range(hp.n_layers)\n        ])\n\n    def forward(\n        self,\n        x: torch.Tensor,             # [B, T, F]\n        source_mask: torch.Tensor,   # [B, T]\n        target_mask: torch.Tensor,   # [B, T]\n        slope: float,\n        chord_change_targets: Optional[torch.Tensor] = None,\n        boundary_teacher_forcing_prob: float = 0.0,\n    ):\n        hp = self.hp\n        B, T, _ = x.shape\n\n        if T != hp.n_steps:\n            raise ValueError(f\"Expected T={hp.n_steps}, got T={T}\")\n\n        source_mask = source_mask.bool()\n        target_mask = target_mask.bool()\n\n        # Encoder input\n        enc_input_embed = self.enc_input_proj(x)\n        enc_input_embed = self.dropout(enc_input_embed)\n        enc_input_embed = self.enc_intra_block(\n            enc_input_embed,\n            n_blocks=hp.n_steps // 4,\n            mask=source_mask,\n        )\n\n        enc_input_embed = enc_input_embed + get_absolute_position_encoding(\n            T, hp.input_embed_size, device=x.device, dtype=x.dtype\n        )\n        enc_input_embed = self.dropout(enc_input_embed)\n\n        # Encoder stack with weighted hidden sum\n        enc_weights = F.softmax(self.enc_layer_weights, dim=0)\n        enc_weighted_hidden = enc_weights[0] * enc_input_embed\n\n        h_enc = enc_input_embed\n        for i, layer in enumerate(self.encoder_layers, start=1):\n            h_enc = layer(h_enc, source_mask)\n            enc_weighted_hidden = enc_weighted_hidden + enc_weights[i] * h_enc\n\n        enc_output = enc_weighted_hidden\n\n        # Boundary prediction\n        chord_change_logits = self.chord_change_head(enc_output).squeeze(-1)   # [B, T]\n        chord_change_prob = torch.sigmoid(slope * chord_change_logits)\n        chord_change_prediction = binary_round(chord_change_prob, cast_to_int=True)\n        chord_change_prediction = chord_change_prediction.clone()\n        chord_change_prediction[:, 0] = 0\n\n        regionalization_changes = chord_change_prediction\n        if (\n            self.training\n            and chord_change_targets is not None\n            and boundary_teacher_forcing_prob > 0.0\n        ):\n            use_teacher = torch.rand((), device=x.device) < boundary_teacher_forcing_prob\n            if bool(use_teacher.item()):\n                regionalization_changes = chord_change_targets.long().clone()\n                regionalization_changes[:, 0] = 0\n\n        # Decoder input\n        dec_input_embed = self.dec_input_proj(x)\n        dec_input_embed = self.dropout(dec_input_embed)\n        dec_input_embed = self.dec_intra_block(\n            dec_input_embed,\n            n_blocks=hp.n_steps // 4,\n            mask=target_mask,\n        )\n\n        # Regionalization\n        dec_input_embed_reg, block_ids, num_blocks = chord_block_compression(\n            dec_input_embed,\n            regionalization_changes,\n            compression=\"mean\",\n        )\n        dec_input_embed_reg = decode_compressed_sequences(dec_input_embed_reg, block_ids)\n        dec_input_embed = dec_input_embed + dec_input_embed_reg + enc_output\n\n        # Decoder PE\n        dec_pe = get_absolute_position_encoding(\n            hp.n_steps, hp.input_embed_size, device=x.device, dtype=x.dtype\n        )\n        dec_pe_batch = dec_pe.expand(B, -1, -1)\n\n        dec_input_embed = dec_input_embed + dec_pe_batch\n        dec_input_embed = self.dropout(dec_input_embed)\n\n        # Decoder stack with weighted hidden sum\n        dec_weights = F.softmax(self.dec_layer_weights, dim=0)\n        dec_weighted_hidden = dec_weights[0] * dec_input_embed\n\n        self_attn_map_list = []\n        attn_map_list = []\n\n        h_dec = dec_input_embed\n        for i, layer in enumerate(self.decoder_layers, start=1):\n            h_dec, self_attn_map, attn_map = layer(\n                x=h_dec,\n                enc=enc_output,\n                dec_pe_batch=dec_pe_batch,\n                target_mask=target_mask,\n                source_mask=source_mask,\n            )\n            self_attn_map_list.append(self_attn_map)\n            attn_map_list.append(attn_map)\n            dec_weighted_hidden = dec_weighted_hidden + dec_weights[i] * h_dec\n\n        dec_output = dec_weighted_hidden\n\n        return {\n            \"chord_change_logits\": chord_change_logits,   # [B, T]\n            \"dec_output\": dec_output,                     # [B, T, C]\n            \"enc_weights\": enc_weights,\n            \"dec_weights\": dec_weights,\n            \"self_attn_maps\": self_attn_map_list,\n            \"attn_maps\": attn_map_list,\n            \"chord_change_prediction\": chord_change_prediction,\n            \"regionalization_changes\": regionalization_changes,\n        }\n\n\n# =========================================================\n# Classifier model\n# =========================================================\n\nclass HTv2ChordModel(nn.Module):\n    \"\"\"\n    Full model:\n      backbone HTv2\n      + chord classification head\n    \"\"\"\n    def __init__(\n        self,\n        input_dim: int,\n        n_chords: int,\n        hyperparameters: HyperParameters,\n        dropout_rate: float = 0.1,\n    ):\n        super().__init__()\n        self.backbone = HTv2(\n            input_dim=input_dim,\n            hyperparameters=hyperparameters,\n            dropout_rate=dropout_rate,\n        )\n        self.chord_classifier = nn.Linear(hyperparameters.input_embed_size, n_chords)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        source_mask: torch.Tensor,\n        target_mask: torch.Tensor,\n        slope: float,\n        chord_change_targets: Optional[torch.Tensor] = None,\n        boundary_teacher_forcing_prob: float = 0.0,\n    ) -> Dict[str, torch.Tensor]:\n        out = self.backbone(\n            x=x,\n            source_mask=source_mask,\n            target_mask=target_mask,\n            slope=slope,\n            chord_change_targets=chord_change_targets,\n            boundary_teacher_forcing_prob=boundary_teacher_forcing_prob,\n        )\n\n        chord_logits = self.chord_classifier(out[\"dec_output\"])  # [B, T, n_chords]\n        out[\"chord_logits\"] = chord_logits\n        return out\n\n\nclass StructuredHTv2ChordModel(nn.Module):\n    \"\"\"\n    HTv2 backbone with six structured chord-component heads.\n\n    The model predicts root+triad, bass, seventh, ninth, eleventh, and\n    thirteenth components. Frame-level full-chord scores are decoded by\n    summing component log-probabilities over the fixed legal chord vocabulary.\n    \"\"\"\n    def __init__(\n        self,\n        input_dim: int,\n        component_sizes: Dict[str, int],\n        chord_component_ids,\n        hyperparameters: HyperParameters,\n        dropout_rate: float = 0.1,\n    ):\n        super().__init__()\n        self.backbone = HTv2(\n            input_dim=input_dim,\n            hyperparameters=hyperparameters,\n            dropout_rate=dropout_rate,\n        )\n        self.component_names = list(component_sizes.keys())\n        self.component_heads = nn.ModuleDict({\n            name: nn.Linear(hyperparameters.input_embed_size, size)\n            for name, size in component_sizes.items()\n        })\n\n        chord_component_ids = torch.as_tensor(chord_component_ids, dtype=torch.long)\n        self.register_buffer(\"chord_component_ids\", chord_component_ids, persistent=False)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        source_mask: torch.Tensor,\n        target_mask: torch.Tensor,\n        slope: float,\n        chord_change_targets: Optional[torch.Tensor] = None,\n        boundary_teacher_forcing_prob: float = 0.0,\n    ) -> Dict[str, torch.Tensor]:\n        out = self.backbone(\n            x=x,\n            source_mask=source_mask,\n            target_mask=target_mask,\n            slope=slope,\n            chord_change_targets=chord_change_targets,\n            boundary_teacher_forcing_prob=boundary_teacher_forcing_prob,\n        )\n\n        component_logits = {\n            name: self.component_heads[name](out[\"dec_output\"])\n            for name in self.component_names\n        }\n\n        chord_scores = None\n        for component_idx, name in enumerate(self.component_names):\n            log_probs = F.log_softmax(component_logits[name], dim=-1)\n            chord_component_ids = self.chord_component_ids[:, component_idx]\n            component_scores = log_probs.index_select(dim=-1, index=chord_component_ids)\n            chord_scores = component_scores if chord_scores is None else chord_scores + component_scores\n\n        out[\"component_logits\"] = component_logits\n        out[\"chord_logits\"] = chord_scores\n        return out\n",
    "models/btc.py": "from __future__ import annotations\n\nfrom typing import Dict, Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .htv2 import CustomLayerNorm, HyperParameters, get_absolute_position_encoding\n\n\nclass BTCDirectionalSelfAttentionBlock(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, dropout_rate: float, direction: str):\n        super().__init__()\n        if direction not in {\"forward\", \"backward\"}:\n            raise ValueError(\"direction must be 'forward' or 'backward'\")\n\n        self.direction = direction\n        self.attn = nn.MultiheadAttention(\n            embed_dim=d_model,\n            num_heads=n_heads,\n            dropout=dropout_rate,\n            batch_first=True,\n        )\n        self.dropout = nn.Dropout(dropout_rate)\n        self.attn_norm = CustomLayerNorm(d_model)\n        self.ffn_norm = CustomLayerNorm(d_model)\n        self.ffn = nn.Sequential(\n            nn.Linear(d_model, d_model),\n            nn.ReLU(),\n            nn.Dropout(dropout_rate),\n            nn.Linear(d_model, d_model),\n            nn.Dropout(dropout_rate),\n        )\n\n    def _attn_mask(self, length: int, device: torch.device) -> torch.Tensor:\n        if self.direction == \"forward\":\n            return torch.triu(torch.ones(length, length, device=device, dtype=torch.bool), diagonal=1)\n        return torch.tril(torch.ones(length, length, device=device, dtype=torch.bool), diagonal=-1)\n\n    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        residual = x\n        x_norm = self.attn_norm(x)\n        attn_output, _ = self.attn(\n            x_norm,\n            x_norm,\n            x_norm,\n            attn_mask=self._attn_mask(x.shape[1], x.device),\n            key_padding_mask=~mask.bool(),\n            need_weights=False,\n        )\n        x = residual + self.dropout(attn_output)\n        x = x + self.ffn(self.ffn_norm(x))\n        return x\n\n\nclass BTCBidirectionalLayer(nn.Module):\n    def __init__(self, d_model: int, n_heads: int, dropout_rate: float):\n        super().__init__()\n        self.forward_block = BTCDirectionalSelfAttentionBlock(\n            d_model=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            direction=\"forward\",\n        )\n        self.backward_block = BTCDirectionalSelfAttentionBlock(\n            d_model=d_model,\n            n_heads=n_heads,\n            dropout_rate=dropout_rate,\n            direction=\"backward\",\n        )\n        self.merge = nn.Linear(d_model * 2, d_model)\n\n    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        forward_out = self.forward_block(x, mask)\n        backward_out = self.backward_block(x, mask)\n        return self.merge(torch.cat([forward_out, backward_out], dim=-1))\n\n\nclass BTCBackbone(nn.Module):\n    def __init__(\n        self,\n        input_dim: int,\n        hyperparameters: HyperParameters,\n        dropout_rate: float = 0.1,\n    ):\n        super().__init__()\n        self.d_model = hyperparameters.input_embed_size\n        self.max_steps = hyperparameters.n_steps\n\n        self.input_dropout = nn.Dropout(dropout_rate)\n        self.input_proj = nn.Linear(input_dim, self.d_model, bias=False)\n        self.layers = nn.ModuleList([\n            BTCBidirectionalLayer(\n                d_model=self.d_model,\n                n_heads=hyperparameters.n_heads,\n                dropout_rate=dropout_rate,\n            )\n            for _ in range(hyperparameters.n_layers)\n        ])\n        self.output_norm = CustomLayerNorm(self.d_model)\n\n    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        y = self.input_dropout(x)\n        y = self.input_proj(y)\n        y = y + get_absolute_position_encoding(\n            length=y.shape[1],\n            hidden_size=self.d_model,\n            device=y.device,\n            dtype=y.dtype,\n        )\n        for layer in self.layers:\n            y = layer(y, mask)\n        return self.output_norm(y)\n\n\nclass BTCChordModel(nn.Module):\n    \"\"\"\n    Pure BTC backbone with the same output interface used by HTv2 training:\n    framewise chord logits + boundary logits.\n    \"\"\"\n    def __init__(\n        self,\n        input_dim: int,\n        n_chords: int,\n        hyperparameters: HyperParameters,\n        dropout_rate: float = 0.1,\n    ):\n        super().__init__()\n        self.backbone = BTCBackbone(\n            input_dim=input_dim,\n            hyperparameters=hyperparameters,\n            dropout_rate=dropout_rate,\n        )\n        self.chord_classifier = nn.Linear(hyperparameters.input_embed_size, n_chords)\n        self.chord_change_head = nn.Linear(hyperparameters.input_embed_size, 1)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        source_mask: torch.Tensor,\n        target_mask: torch.Tensor,\n        slope: float,\n        chord_change_targets: Optional[torch.Tensor] = None,\n        boundary_teacher_forcing_prob: float = 0.0,\n    ) -> Dict[str, torch.Tensor]:\n        del target_mask, slope, chord_change_targets, boundary_teacher_forcing_prob\n\n        enc_output = self.backbone(x=x, mask=source_mask)\n        return {\n            \"enc_output\": enc_output,\n            \"chord_logits\": self.chord_classifier(enc_output),\n            \"chord_change_logits\": self.chord_change_head(enc_output).squeeze(-1),\n        }\n\n\nclass StructuredBTCChordModel(nn.Module):\n    \"\"\"\n    Pure BTC backbone with the same structured full-chord objective used by HTv2.\n    \"\"\"\n    def __init__(\n        self,\n        input_dim: int,\n        component_sizes: Dict[str, int],\n        chord_component_ids,\n        hyperparameters: HyperParameters,\n        dropout_rate: float = 0.1,\n    ):\n        super().__init__()\n        self.backbone = BTCBackbone(\n            input_dim=input_dim,\n            hyperparameters=hyperparameters,\n            dropout_rate=dropout_rate,\n        )\n        self.component_names = list(component_sizes.keys())\n        self.component_heads = nn.ModuleDict({\n            name: nn.Linear(hyperparameters.input_embed_size, size)\n            for name, size in component_sizes.items()\n        })\n        self.chord_change_head = nn.Linear(hyperparameters.input_embed_size, 1)\n\n        chord_component_ids = torch.as_tensor(chord_component_ids, dtype=torch.long)\n        self.register_buffer(\"chord_component_ids\", chord_component_ids, persistent=False)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        source_mask: torch.Tensor,\n        target_mask: torch.Tensor,\n        slope: float,\n        chord_change_targets: Optional[torch.Tensor] = None,\n        boundary_teacher_forcing_prob: float = 0.0,\n    ) -> Dict[str, torch.Tensor]:\n        del target_mask, slope, chord_change_targets, boundary_teacher_forcing_prob\n\n        enc_output = self.backbone(x=x, mask=source_mask)\n        component_logits = {\n            name: self.component_heads[name](enc_output)\n            for name in self.component_names\n        }\n\n        chord_scores = None\n        for component_idx, name in enumerate(self.component_names):\n            log_probs = F.log_softmax(component_logits[name], dim=-1)\n            component_ids = self.chord_component_ids[:, component_idx]\n            component_scores = log_probs.index_select(dim=-1, index=component_ids)\n            chord_scores = component_scores if chord_scores is None else chord_scores + component_scores\n\n        return {\n            \"enc_output\": enc_output,\n            \"component_logits\": component_logits,\n            \"chord_logits\": chord_scores,\n            \"chord_change_logits\": self.chord_change_head(enc_output).squeeze(-1),\n        }\n",
}
for rel_path, content in EMBEDDED_FILES.items():
    out_path = RUNTIME_DIR / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(content, encoding='utf-8')

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print(f'Runtime source dir: {RUNTIME_DIR}')
print(sorted(str(p.relative_to(RUNTIME_DIR)) for p in RUNTIME_DIR.rglob('*.py')))


In [ ]:

from evaluation_utils import (
    EvalCheckpointConfig,
    apply_feature_noise_snr,
    apply_feature_noise_std,
    apply_signal_decay,
    evaluate_large_vocabulary,
    evaluate_mir_eval_metrics,
    export_predictions,
    load_eval_model,
    resolve_song_ids,
)


In [ ]:

from pathlib import Path
import math
import torch

ROOT_DIR = (DRIVE_PROJECT_DIR / 'data' / 'chord_data_1217').resolve()
CHECKPOINT_PATH = ROOT_DIR / 'checkpoints' / 'YOUR_EXPERIMENT' / 'fold_0_best.pt'
MODEL_TYPE = 'btc'  # 'btc' or 'htv2'
# Example checkpoint path: ROOT_DIR / 'checkpoints' / 'compare_btc_fold0' / 'fold_0_best.pt'
LABEL_MODE = 'structured_full_chord'  # 'full_chord' or 'structured_full_chord'
FOLD_ID = 0
SPLIT = 'test'  # 'train', 'val', 'test', or 'all'

# These hyperparameters must match the checkpoint you trained.
EMBED_SIZE = 256
N_LAYERS = 4
N_HEADS = 8
DROPOUT = 0.3
N_STEPS = 128
STRIDE = 64

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WINDOW_BATCH_SIZE = 32
OVERWRITE_PREDICTIONS = False
LIMIT_SONGS = None  # set to a small int for a quick smoke test

# Corruption presets. Choose one mode.
EVAL_MODE = 'signal_decay'  # 'signal_decay', 'feature_noise_snr', or 'feature_noise_std'
if EVAL_MODE == 'signal_decay':
    LEVELS = {
        'clean': 1.0,
        'little': 0.85,
        'some': 0.60,
        'lots': 0.35,
    }
elif EVAL_MODE == 'feature_noise_snr':
    LEVELS = {
        'clean': math.inf,
        'little': 20.0,
        'some': 10.0,
        'lots': 0.0,
    }
elif EVAL_MODE == 'feature_noise_std':
    LEVELS = {
        'clean': 0.0,
        'little': 0.01,
        'some': 0.03,
        'lots': 0.08,
    }
else:
    raise ValueError(f'Unsupported EVAL_MODE: {EVAL_MODE}')

OUTPUT_ROOT = (DRIVE_PROJECT_DIR / 'eval_outputs' / CHECKPOINT_PATH.stem / 'colab_portable' / (EVAL_MODE) / f'fold_{FOLD_ID}_{SPLIT}').resolve()
REF_DIR = (ROOT_DIR / 'chordlab').resolve()

assert ROOT_DIR.exists(), f'Missing root dir: {ROOT_DIR}'
assert CHECKPOINT_PATH.exists(), f'Missing checkpoint: {CHECKPOINT_PATH}'
assert REF_DIR.exists(), f'Missing reference lab dir: {REF_DIR}'
assert LABEL_MODE in {'full_chord', 'structured_full_chord'}

print(f'Root dir: {ROOT_DIR}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Output root: {OUTPUT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Mode: {EVAL_MODE} -> {LEVELS}')


In [ ]:

cfg = EvalCheckpointConfig(
    root_dir=str(ROOT_DIR),
    checkpoint_path=str(CHECKPOINT_PATH),
    model_type=MODEL_TYPE,
    label_mode=LABEL_MODE,
    fold_id=FOLD_ID,
    split=SPLIT,
    embed_size=EMBED_SIZE,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    n_steps=N_STEPS,
    stride=STRIDE,
    device=DEVICE,
)

model, vocab, device = load_eval_model(cfg)
print(type(model).__name__)
print(f'Vocab size: {vocab.size}')
print(f'Songs in split: {len(resolve_song_ids(str(ROOT_DIR), FOLD_ID, SPLIT))}')


## Export predictions

Each level writes `.lab` files into `eval_outputs/.../<level>/`.

`signal_decay` multiplies the feature sequence by a linear envelope from `1.0` to `final_gain`.
`feature_noise_snr` and `feature_noise_std` add Gaussian noise directly to the processed CQT features.


In [ ]:

from pathlib import Path

summaries = {}

for level_name, level_value in LEVELS.items():
    level_dir = OUTPUT_ROOT / level_name

    def corruption_fn(x, song_id, *, _mode=EVAL_MODE, _value=level_value, _level_name=level_name):
        if _mode == 'signal_decay':
            return x if _value == 1.0 else apply_signal_decay(x, final_gain=_value)
        seed = abs(hash((song_id, _level_name))) % (2**32)
        if _mode == 'feature_noise_snr':
            return apply_feature_noise_snr(x, snr_db=_value, seed=seed)
        if _mode == 'feature_noise_std':
            return apply_feature_noise_std(x, noise_std=_value, seed=seed)
        raise ValueError(_mode)

    corruption = None if level_name == 'clean' else corruption_fn
    summary = export_predictions(
        cfg=cfg,
        output_dir=str(level_dir),
        corruption_fn=corruption,
        batch_size=WINDOW_BATCH_SIZE,
        overwrite=OVERWRITE_PREDICTIONS,
        limit=LIMIT_SONGS,
    )
    summaries[level_name] = summary
    print(level_name, summary)


## Score exported `.lab` files

`frame_acc` and `class_acc` use seen-only class averaging, matching the teammate notebook logic.


In [ ]:

import pandas as pd

rows = []
vocab_labels = list(vocab.idx_to_chord)

for level_name in LEVELS:
    est_dir = OUTPUT_ROOT / level_name
    mir_scores = evaluate_mir_eval_metrics(str(est_dir), str(REF_DIR))
    large_vocab = evaluate_large_vocabulary(
        est_dir=str(est_dir),
        ref_dir=str(REF_DIR),
        vocab_labels=vocab_labels,
        fps=22050.0 / 512.0,
    )
    rows.append({
        'level': level_name,
        'model_type': MODEL_TYPE,
        'label_mode': LABEL_MODE,
        'frame_acc': large_vocab['frame_acc'],
        'class_acc': large_vocab['class_acc'],
        'classes_seen': large_vocab['classes_seen'],
        'vocab_keys': large_vocab['vocab_key_count'],
        **mir_scores,
    })

results_df = pd.DataFrame(rows)
results_df = results_df[[
    'level',
    'model_type',
    'label_mode',
    'frame_acc',
    'class_acc',
    'classes_seen',
    'vocab_keys',
    'root',
    'thirds',
    'majmin',
    'triads',
    'sevenths',
    'tetrads',
    'mirex',
]]
results_df


In [ ]:

results_csv = OUTPUT_ROOT / 'results.csv'
results_df.to_csv(results_csv, index=False)
print(f'Saved: {results_csv}')
